# LLM-Aided Testbench Generation — Homework

**Student:** Sahil Dunani (sd6429)
**Course:** LLM4ChipDesign — Enhanced LLM-Aided Testbench Generation
**Instructors:** Prof. Ramesh Karri, Dr. Weihua Xiao

This notebook implements the four-step LLM-driven testbench generation pipeline described in the assignment PDF:

1. **Initial testbench** — the LLM reads a natural-language spec and generates a Verilog testbench with ~20 diverse stimulus patterns (corner cases + randomized).
2. **Golden model** — the LLM reads the same spec and writes a Python `golden_model.py` that implements the expected behavior.
3. **Patterns with golden outputs** — Python runs the stimulus patterns through the golden model to produce `test_patterns_with_golden.json`.
4. **Final self-checking testbench** — the LLM updates the initial TB to compare DUT outputs against the stored golden values and prints PASS / FAIL per test.

For each example we emit the four artifacts the assignment requires: `testbench_initial.v`, `golden_model.py`, `test_patterns_with_golden.json`, `testbench_final.v`.

**Parts covered:** I (mux + adder tutorials), II (inspect-and-explain for the adder), III (custom module — saturating adder), IV (bug-detection demo).

## Setup

In [31]:
# Install deps
!pip install --quiet openai
!apt-get -qq update && apt-get -qq install -y iverilog

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [32]:
# API key — read from Colab Secrets, NEVER hardcoded in the notebook
from openai import OpenAI
from google.colab import userdata
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
print("OpenAI client ready.")

OpenAI client ready.


## Pipeline framework

A single function `run_pipeline(name, spec, module_name, port_decl, dut_verilog)` runs the complete 4-step pipeline for one example. It writes all four required artifacts to a per-example subfolder and verifies with `iverilog` + `vvp`.

In [33]:
import os, re, json, subprocess, textwrap, importlib.util, random
from typing import Dict, Any, List, Tuple

MODEL = "gpt-4o-mini"
TEMP  = 0.3
MAX_TOKENS = 2500

def ask_llm(messages, max_tokens=MAX_TOKENS, temperature=TEMP):
    resp = client.chat.completions.create(
        model=MODEL, temperature=temperature, max_tokens=max_tokens,
        messages=messages, stream=False,
    )
    return resp.choices[0].message.content

VBLOCK = re.compile(r"```(?:verilog|systemverilog|sv)?\s*\n(.*?)```", re.S)
PBLOCK = re.compile(r"```python\s*\n(.*?)```", re.S)
JBLOCK = re.compile(r"```json\s*\n(.*?)```", re.S)

def extract(pat, text, fallback_token=None):
    m = pat.search(text)
    if m: return m.group(1).strip()
    if fallback_token and fallback_token in text:
        return text[text.index(fallback_token):].strip()
    return text.strip()

def run(cmd, cwd, timeout=60):
    try:
        p = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True,
                           text=True, timeout=timeout)
        return p.returncode, (p.stdout or "") + (p.stderr or "")
    except subprocess.TimeoutExpired:
        return -1, f"TIMEOUT after {timeout}s"

# ----- Step 1: initial testbench + the pattern list it uses -----
def generate_initial_tb(spec: str, module_name: str, port_decl: str,
                        pattern_count_hint: str = "about 20") -> Tuple[str, List[Dict[str, int]]]:
    prompt = textwrap.dedent(f"""\
    You are generating a stimulus-only Verilog testbench for Icarus Verilog.

    MODULE SPECIFICATION:
    {spec}

    MODULE PORT DECLARATION (DUT):
    ```verilog
    {port_decl}
    ```

    Produce TWO things:

    (A) A complete Verilog testbench module named `tb_{module_name}` that:
      - Instantiates the DUT as `dut` with named port connections.
      - Applies {pattern_count_hint} diverse test patterns including BOTH boundary / corner cases
        (zeros, max values, all-ones) AND randomized / mid-range cases.
      - For every pattern, after applying stimulus, waits #1 and uses `$display` with EXACTLY this prefix
        format: `TEST <n>: <input_name>=<val> ... <output_name>=<val> ...` (space-separated name=val tokens).
      - Does NOT include any checking against expected values — purely stimulus + print.
      - Ends with `$finish;`.

    (B) A JSON object with the EXACT patterns the testbench uses, in the SAME order, formatted as:
      ```json
      {{"patterns": [
        {{"<input_name>": <int>, ...}},
        ...
      ]}}
      ```
      Only input signals, no outputs.

    Return the Verilog in a ```verilog block and the JSON in a ```json block. No prose outside those blocks.
    """).strip()

    reply = ask_llm([{"role": "user", "content": prompt}])
    tb_v = extract(VBLOCK, reply, fallback_token="module ")
    pats_text = extract(JBLOCK, reply, fallback_token="{")
    try:
        pats = json.loads(pats_text)["patterns"]
    except Exception:
        # retry parsing by locating braces
        s = pats_text.find("{"); e = pats_text.rfind("}") + 1
        pats = json.loads(pats_text[s:e])["patterns"]
    return tb_v, pats

# ----- Step 2: Python golden model -----
def generate_golden_model(spec: str, module_name: str, port_decl: str) -> str:
    prompt = textwrap.dedent(f"""\
    Write a Python golden reference model for a Verilog module.

    MODULE SPECIFICATION:
    {spec}

    MODULE PORT DECLARATION:
    ```verilog
    {port_decl}
    ```

    Produce a single Python file that defines ONE function:

        def oracle(inputs: dict) -> dict:
            ...

    - `inputs` is a dict mapping each INPUT port name to an integer value.
    - Return a dict mapping each OUTPUT port name to its correct integer value, as per the spec.
    - Implement the behavior described in the spec faithfully.
    - Use only the Python standard library; no external imports needed beyond what Python 3 ships with.
    - Do NOT include test code, prints, or `if __name__ == "__main__"` blocks — just the function.

    Return ONLY one ```python code block. No prose.
    """).strip()

    reply = ask_llm([{"role": "user", "content": prompt}])
    return extract(PBLOCK, reply, fallback_token="def oracle")

# ----- Step 3: run patterns through the golden model -----
def apply_golden(patterns: List[Dict[str, int]], golden_code: str,
                 example_dir: str) -> List[Dict[str, Any]]:
    gpath = os.path.join(example_dir, "golden_model.py")
    open(gpath, "w").write(golden_code)
    spec = importlib.util.spec_from_file_location("golden_model", gpath)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    out = []
    for p in patterns:
        exp = mod.oracle(dict(p))
        out.append({"inputs": p, "expected": exp})
    return out

# ----- Step 4: final self-checking testbench -----
def generate_final_tb(initial_tb: str, patterns_with_golden: List[Dict[str, Any]],
                      module_name: str) -> str:
    prompt = textwrap.dedent(f"""\
    Here is an initial stimulus-only Verilog testbench:
    ```verilog
    {initial_tb}
    ```

    Here are the same patterns with the expected output values computed by the Python golden model:
    ```json
    {json.dumps(patterns_with_golden, indent=2)}
    ```

    Produce a self-checking Verilog testbench called `tb_{module_name}` that:
      - Applies the SAME patterns in the SAME order.
      - Declares `integer errors = 0;` and `integer test_num = 0;`.
      - For each pattern, after waiting #1, compares each DUT output signal to the stored
        expected value. Use `!==` for the compare.
      - On MATCH print: `TEST <n> PASS: <input_name>=<val> ... <output_name>=<val>`
      - On MISMATCH print: `TEST <n> FAIL: <input_name>=<val> ... expected <output_name>=<exp> got=<actual>`
        and increment `errors`.
      - At the end print either `ALL TESTS PASSED` (if errors == 0) or `<N> TEST(S) FAILED` (otherwise).
      - End with `$finish;`.
      - Do NOT read the JSON file at runtime — hardcode the expected values alongside the stimulus.
      - Use the same ports / widths as the original testbench.

    Return ONLY the Verilog testbench in one ```verilog block.
    """).strip()

    reply = ask_llm([{"role": "user", "content": prompt}])
    return extract(VBLOCK, reply, fallback_token="module ")

# ----- end-to-end -----
def run_pipeline(name: str, spec: str, module_name: str, port_decl: str,
                 dut_verilog: str, verbose: bool = True) -> Dict[str, Any]:
    """Run the 4-step pipeline for one example. Returns dict with artifact paths and sim result."""
    example_dir = os.path.join(".", name)
    os.makedirs(example_dir, exist_ok=True)

    # Save DUT
    dut_path = os.path.join(example_dir, f"{module_name}.v")
    open(dut_path, "w").write(dut_verilog)
    if verbose: print(f"[{name}] DUT  -> {dut_path}")

    # Step 1
    if verbose: print(f"[{name}] Step 1: generating initial testbench + patterns ...")
    initial_tb, patterns = generate_initial_tb(spec, module_name, port_decl)
    initial_path = os.path.join(example_dir, "testbench_initial.v")
    open(initial_path, "w").write(initial_tb)
    if verbose: print(f"[{name}]        {len(patterns)} patterns, TB -> {initial_path}")

    # Step 2
    if verbose: print(f"[{name}] Step 2: generating Python golden model ...")
    golden_code = generate_golden_model(spec, module_name, port_decl)
    # (apply_golden writes golden_model.py too)

    # Step 3
    if verbose: print(f"[{name}] Step 3: running patterns through golden model ...")
    patterns_with_golden = apply_golden(patterns, golden_code, example_dir)
    pwg_path = os.path.join(example_dir, "test_patterns_with_golden.json")
    with open(pwg_path, "w") as f:
        json.dump({"patterns": patterns_with_golden}, f, indent=2)
    if verbose: print(f"[{name}]        -> {pwg_path}")

    # Step 4
    if verbose: print(f"[{name}] Step 4: generating final self-checking testbench ...")
    final_tb = generate_final_tb(initial_tb, patterns_with_golden, module_name)
    final_path = os.path.join(example_dir, "testbench_final.v")
    open(final_path, "w").write(final_tb)
    if verbose: print(f"[{name}]        -> {final_path}")

    # Verify
    if verbose: print(f"[{name}] Verifying with iverilog + vvp ...")
    rc, out_c = run(f"iverilog -g2012 -o tb.vvp {module_name}.v testbench_final.v",
                    cwd=example_dir, timeout=45)
    if rc != 0:
        print(f"[{name}] COMPILE FAILED:\n{out_c[-800:]}")
        return {"name": name, "compiled": False, "passed": False, "sim": out_c}

    rc_s, out_s = run("vvp tb.vvp", cwd=example_dir, timeout=90)
    passed = ("ALL TESTS PASSED" in out_s) and ("FAIL" not in out_s)
    print(f"[{name}] {'>>> PASS' if passed else '>>> FAIL / partial'}")
    print(out_s.strip()[-1200:])

    return {
        "name": name, "example_dir": example_dir,
        "compiled": True, "passed": passed, "num_patterns": len(patterns),
        "sim": out_s,
        "artifacts": {
            "testbench_initial": initial_path,
            "golden_model": os.path.join(example_dir, "golden_model.py"),
            "patterns_with_golden": pwg_path,
            "testbench_final": final_path,
            "dut": dut_path,
        }
    }


---
# Part I — Two tutorial examples

The assignment allows picking any two of the three tutorial examples. I ran the **2-to-1 multiplexer** (combinational) and the **4-bit adder** (arithmetic).

## Example 1 — 2-to-1 Multiplexer

**Spec:** A combinational 2-to-1 multiplexer. Two 1-bit data inputs `a` and `b`, a 1-bit select `sel`. The output `y` equals `a` when `sel=0`, else `b`.

In [34]:
MUX_SPEC = """A 1-bit 2-to-1 multiplexer. Inputs: a (1-bit), b (1-bit), sel (1-bit).
Output: y (1-bit). Behavior: when sel is 0 the output y equals a; when sel is 1 the output y equals b.
Purely combinational, no clock, no reset."""

MUX_PORT_DECL = """module mux21(
    input  wire a,
    input  wire b,
    input  wire sel,
    output wire y
);"""

MUX_DUT_V = """module mux21(input wire a, input wire b, input wire sel, output wire y);
    assign y = sel ? b : a;
endmodule"""

result_mux = run_pipeline("mux", MUX_SPEC, "mux21", MUX_PORT_DECL, MUX_DUT_V)

[mux] DUT  -> ./mux/mux21.v
[mux] Step 1: generating initial testbench + patterns ...
[mux]        20 patterns, TB -> ./mux/testbench_initial.v
[mux] Step 2: generating Python golden model ...
[mux] Step 3: running patterns through golden model ...
[mux]        -> ./mux/test_patterns_with_golden.json
[mux] Step 4: generating final self-checking testbench ...
[mux]        -> ./mux/testbench_final.v
[mux] Verifying with iverilog + vvp ...
[mux] COMPILE FAILED:
testbench_final.v:138: syntax error
I give up.



## Example 2 — 4-bit Adder

**Spec:** A combinational 4-bit adder with carry-in and carry-out. Two 4-bit operands `a` and `b`, 1-bit `cin`, 4-bit sum output and 1-bit carry-out.

In [35]:
ADDER_SPEC = """A 4-bit ripple-carry adder with carry-in and carry-out.
Inputs: a (4-bit), b (4-bit), cin (1-bit). Outputs: sum (4-bit), cout (1-bit).
Behavior: {cout, sum} = a + b + cin (so cout is the overflow bit from a 5-bit addition).
Purely combinational, no clock, no reset."""

ADDER_PORT_DECL = """module adder4(
    input  wire [3:0] a,
    input  wire [3:0] b,
    input  wire       cin,
    output wire [3:0] sum,
    output wire       cout
);"""

ADDER_DUT_V = """module adder4(input wire [3:0] a, input wire [3:0] b, input wire cin,
                               output wire [3:0] sum, output wire cout);
    assign {cout, sum} = a + b + cin;
endmodule"""

result_adder = run_pipeline("adder", ADDER_SPEC, "adder4", ADDER_PORT_DECL, ADDER_DUT_V)

[adder] DUT  -> ./adder/adder4.v
[adder] Step 1: generating initial testbench + patterns ...
[adder]        20 patterns, TB -> ./adder/testbench_initial.v
[adder] Step 2: generating Python golden model ...
[adder] Step 3: running patterns through golden model ...
[adder]        -> ./adder/test_patterns_with_golden.json
[adder] Step 4: generating final self-checking testbench ...
[adder]        -> ./adder/testbench_final.v
[adder] Verifying with iverilog + vvp ...
[adder] COMPILE FAILED:
testbench_final.v:147: syntax error
I give up.



---
# Part II — Inspect and explain the generated artifacts (adder)

For the 4-bit adder, let's look at the three artifacts the pipeline produced and show one concrete before/after diff between the initial and final testbenches.

### The golden model (`golden_model.py`)

The LLM generated a Python `oracle(inputs: dict) -> dict` that computes the expected 5-bit sum and splits it into `sum` (lower 4 bits) and `cout` (bit 4). This is what we execute on every pattern to produce the reference outputs.

In [36]:
print(open("adder/golden_model.py").read())

def oracle(inputs: dict) -> dict:
    a = inputs['a']
    b = inputs['b']
    cin = inputs['cin']
    
    total = a + b + cin
    sum_result = total & 0b1111  # Get the lower 4 bits for sum
    cout_result = (total >> 4) & 0b1  # Get the 5th bit for carry out
    
    return {
        'sum': sum_result,
        'cout': cout_result
    }


### The patterns-with-golden JSON (`test_patterns_with_golden.json`)

Each entry has `inputs` (what we drive on the DUT) and `expected` (what the golden model computes). Showing the first 5 for brevity:

In [37]:
import json
pwg = json.load(open("adder/test_patterns_with_golden.json"))
print(f"Total patterns: {len(pwg['patterns'])}\n")
for i, p in enumerate(pwg["patterns"][:5]):
    print(f"[{i}] inputs={p['inputs']}  expected={p['expected']}")

Total patterns: 20

[0] inputs={'a': 0, 'b': 0, 'cin': 0}  expected={'sum': 0, 'cout': 0}
[1] inputs={'a': 1, 'b': 1, 'cin': 0}  expected={'sum': 2, 'cout': 0}
[2] inputs={'a': 2, 'b': 3, 'cin': 0}  expected={'sum': 5, 'cout': 0}
[3] inputs={'a': 15, 'b': 1, 'cin': 0}  expected={'sum': 0, 'cout': 1}
[4] inputs={'a': 15, 'b': 15, 'cin': 0}  expected={'sum': 14, 'cout': 1}


### Updater effect — initial vs. final testbench

The initial testbench only drives inputs and prints what it sees; the final one adds expected values and compare / pass / fail logic. Showing the first ~25 lines of each so the diff is obvious:

In [38]:
def head(path, n=25):
    for i, ln in enumerate(open(path).read().splitlines()):
        if i >= n: break
        print(ln)

print("=== testbench_initial.v (first 25 lines) ===")
head("adder/testbench_initial.v")
print()
print("=== testbench_final.v (first 25 lines) ===")
head("adder/testbench_final.v")

=== testbench_initial.v (first 25 lines) ===
module tb_adder4;

    reg [3:0] a;
    reg [3:0] b;
    reg cin;
    wire [3:0] sum;
    wire cout;

    // Instantiate the DUT
    adder4 dut (
        .a(a),
        .b(b),
        .cin(cin),
        .sum(sum),
        .cout(cout)
    );

    initial begin
        // Test patterns
        a = 4'b0000; b = 4'b0000; cin = 1'b0; #1;
        $display("TEST 1: a=%b b=%b cin=%b sum=%b cout=%b", a, b, cin, sum, cout);
        
        a = 4'b0001; b = 4'b0001; cin = 1'b0; #1;
        $display("TEST 2: a=%b b=%b cin=%b sum=%b cout=%b", a, b, cin, sum, cout);
        

=== testbench_final.v (first 25 lines) ===
module tb_adder4;

    reg [3:0] a;
    reg [3:0] b;
    reg cin;
    wire [3:0] sum;
    wire cout;

    // Instantiate the DUT
    adder4 dut (
        .a(a),
        .b(b),
        .cin(cin),
        .sum(sum),
        .cout(cout)
    );

    integer errors = 0;
    integer test_num = 0;

    initial begin
        // Test patterns with e

**What changed:** the final TB added an expected-value table (either `localparam` constants or an inline `if (sum !== <exp>)` compare per test), an `integer errors` counter, pass/fail `$display` messages, and a final summary (`ALL TESTS PASSED` or `<N> TEST(S) FAILED`). The initial TB had none of that — only stimulus + a plain print of whatever the DUT returned.

---
# Part III — Custom module: 4-bit saturating adder

**Design choice:** a 4-bit saturating adder — the computed sum is clamped at `4'hF` (15) when `a + b` would otherwise overflow. This has a clean interface (two 4-bit inputs, one 4-bit output), interesting corner cases (0+0, 15+0, 8+8, 15+15, every pair that overflows), and the saturation behavior is easy to describe and model.

**Why it's a good test:** a normal adder would wrap on overflow; a saturating adder clips to the max. The two differ *only* on overflow cases, so a mistake in the saturation check is both easy to introduce (Part IV uses this) and clearly caught by well-chosen test patterns.

In [39]:
SATADD_SPEC = """A 4-bit saturating adder. Inputs: a (4-bit unsigned), b (4-bit unsigned).
Output: sum (4-bit unsigned). Behavior: compute the 5-bit arithmetic sum of a and b.
If the 5-bit sum is less than or equal to 15 (i.e. no overflow), output that value in `sum`.
If the 5-bit sum is greater than 15 (overflow), SATURATE the output to 4'd15 (all ones).
Purely combinational, no clock, no reset.

Examples of saturating behavior:
- a=4'd0, b=4'd0 -> sum = 0
- a=4'd7, b=4'd5 -> sum = 12
- a=4'd15, b=4'd0 -> sum = 15
- a=4'd8, b=4'd8 -> sum = 15 (saturates, raw sum would be 16)
- a=4'd15, b=4'd15 -> sum = 15 (saturates, raw sum would be 30)
"""

SATADD_PORT_DECL = """module sat_adder4(
    input  wire [3:0] a,
    input  wire [3:0] b,
    output wire [3:0] sum
);"""

# Golden implementation (what Part IV will mutate).
SATADD_DUT_V = """module sat_adder4(input wire [3:0] a, input wire [3:0] b, output wire [3:0] sum);
    wire [4:0] raw;
    assign raw = a + b;
    assign sum = (raw > 5'd15) ? 4'd15 : raw[3:0];
endmodule"""

result_satadd = run_pipeline("sat_adder", SATADD_SPEC, "sat_adder4", SATADD_PORT_DECL, SATADD_DUT_V)
print(f"\nGenerated {result_satadd['num_patterns']} patterns.")

[sat_adder] DUT  -> ./sat_adder/sat_adder4.v
[sat_adder] Step 1: generating initial testbench + patterns ...
[sat_adder]        20 patterns, TB -> ./sat_adder/testbench_initial.v
[sat_adder] Step 2: generating Python golden model ...
[sat_adder] Step 3: running patterns through golden model ...
[sat_adder]        -> ./sat_adder/test_patterns_with_golden.json
[sat_adder] Step 4: generating final self-checking testbench ...
[sat_adder]        -> ./sat_adder/testbench_final.v
[sat_adder] Verifying with iverilog + vvp ...
[sat_adder] >>> PASS
TEST           1 PASS: a= 0 b= 0 sum= 0
TEST           2 PASS: a= 7 b= 5 sum=12
TEST           3 PASS: a=15 b= 0 sum=15
TEST           4 PASS: a= 8 b= 8 sum=15
TEST           5 PASS: a=15 b=15 sum=15
TEST           6 PASS: a= 1 b= 1 sum= 2
TEST           7 PASS: a= 3 b= 2 sum= 5
TEST           8 PASS: a= 4 b= 4 sum= 8
TEST           9 PASS: a= 2 b= 3 sum= 5
TEST          10 PASS: a=10 b= 6 sum=15
TEST          11 PASS: a=14 b= 3 sum=15
TEST          1

---
# Part IV — Bug detection demo

To show that the generated self-checking testbench actually catches implementation bugs, I take the working saturating-adder DUT from Part III and inject a small intentional bug: **remove the saturation check**, so overflowing sums wrap around (as a plain 4-bit adder would) instead of saturating at 15.

Expected result: patterns where `a + b <= 15` will still pass, but patterns where `a + b > 15` will FAIL with the actual (wrapped) value differing from the expected (15).

Then I restore the correct DUT and re-run — all patterns pass.

## Step 1 — inject the bug

In [40]:
BUGGY_SATADD = """module sat_adder4(input wire [3:0] a, input wire [3:0] b, output wire [3:0] sum);
    // BUG INJECTED (Part IV): no saturation. Overflow wraps instead of clipping.
    assign sum = a + b;
endmodule"""

open("sat_adder/sat_adder4.v", "w").write(BUGGY_SATADD)
print("Buggy DUT written. Re-running the SAME testbench_final.v against the buggy DUT ...\n")

import subprocess
rc_c, out_c = subprocess.run(
    "iverilog -g2012 -o tb.vvp sat_adder4.v testbench_final.v",
    shell=True, cwd="sat_adder", capture_output=True, text=True
).returncode, ""
# re-compile + re-run cleanly
_rc, _out = run("iverilog -g2012 -o tb.vvp sat_adder4.v testbench_final.v && vvp tb.vvp",
                cwd="sat_adder", timeout=60)
print(_out)

Buggy DUT written. Re-running the SAME testbench_final.v against the buggy DUT ...

TEST           1 PASS: a= 0 b= 0 sum= 0
TEST           2 PASS: a= 7 b= 5 sum=12
TEST           3 PASS: a=15 b= 0 sum=15
TEST           4 FAIL: a= 8 b= 8 expected sum=15 got= 0
TEST           5 FAIL: a=15 b=15 expected sum=15 got=14
TEST           6 PASS: a= 1 b= 1 sum= 2
TEST           7 PASS: a= 3 b= 2 sum= 5
TEST           8 PASS: a= 4 b= 4 sum= 8
TEST           9 PASS: a= 2 b= 3 sum= 5
TEST          10 FAIL: a=10 b= 6 expected sum=15 got= 0
TEST          11 FAIL: a=14 b= 3 expected sum=15 got= 1
TEST          12 PASS: a= 5 b=10 sum=15
TEST          13 FAIL: a=12 b= 4 expected sum=15 got= 0
TEST          14 PASS: a= 6 b= 9 sum=15
TEST          15 FAIL: a=13 b= 3 expected sum=15 got= 0
TEST          16 PASS: a= 0 b=15 sum=15
TEST          17 FAIL: a=15 b= 1 expected sum=15 got= 0
TEST          18 FAIL: a= 7 b= 9 expected sum=15 got= 0
TEST          19 FAIL: a=11 b= 5 expected sum=15 got= 0
TEST        

Above you can see the testbench prints several `TEST N FAIL: ...` lines wherever the raw sum exceeds 15 (overflow). The final line reports some number of failing tests — this is the testbench successfully detecting the bug.

## Step 2 — restore the correct DUT and re-run

In [41]:
# Restore the correct saturating adder
open("sat_adder/sat_adder4.v", "w").write(SATADD_DUT_V)
print("Correct DUT restored. Re-running testbench_final.v ...\n")

_rc, _out = run("iverilog -g2012 -o tb.vvp sat_adder4.v testbench_final.v && vvp tb.vvp",
                cwd="sat_adder", timeout=60)
print(_out)

Correct DUT restored. Re-running testbench_final.v ...

TEST           1 PASS: a= 0 b= 0 sum= 0
TEST           2 PASS: a= 7 b= 5 sum=12
TEST           3 PASS: a=15 b= 0 sum=15
TEST           4 PASS: a= 8 b= 8 sum=15
TEST           5 PASS: a=15 b=15 sum=15
TEST           6 PASS: a= 1 b= 1 sum= 2
TEST           7 PASS: a= 3 b= 2 sum= 5
TEST           8 PASS: a= 4 b= 4 sum= 8
TEST           9 PASS: a= 2 b= 3 sum= 5
TEST          10 PASS: a=10 b= 6 sum=15
TEST          11 PASS: a=14 b= 3 sum=15
TEST          12 PASS: a= 5 b=10 sum=15
TEST          13 PASS: a=12 b= 4 sum=15
TEST          14 PASS: a= 6 b= 9 sum=15
TEST          15 PASS: a=13 b= 3 sum=15
TEST          16 PASS: a= 0 b=15 sum=15
TEST          17 PASS: a=15 b= 1 sum=15
TEST          18 PASS: a= 7 b= 9 sum=15
TEST          19 PASS: a=11 b= 5 sum=15
TEST          20 PASS: a= 9 b= 7 sum=15
ALL TESTS PASSED
testbench_final.v:43: $finish called at 20 (1s)



Final line should now read `ALL TESTS PASSED` — the testbench matches the DUT after the fix.

---
# Summary

In [46]:
print("Pipeline summary")
print("-" * 60)
for r in [result_mux, result_adder, result_satadd]:
    passed = r.get("passed", False)
    compiled = r.get("compiled", False)
    if not compiled:
        status = "COMPILE FAIL"
    elif passed:
        status = "PASS"
    else:
        status = "SIM FAIL"
    print(f"  {r['name']:<12} {status:<14} patterns={r.get('num_patterns', '?'):<3} dir={r.get('example_dir','?')}")
print()
print("Artifacts generated per example:")
print("  - <example>/testbench_initial.v         (step 1 - stimulus only)")
print("  - <example>/golden_model.py             (step 2 - Python oracle)")
print("  - <example>/test_patterns_with_golden.json (step 3 - oracle run on each pattern)")
print("  - <example>/testbench_final.v           (step 4 - self-checking TB)")
print("  - <example>/<module>.v                  (DUT being verified)")

Pipeline summary
------------------------------------------------------------
  mux          PASS           patterns=20  dir=./mux
  adder        PASS           patterns=20  dir=./adder
  sat_adder    PASS           patterns=20  dir=./sat_adder

Artifacts generated per example:
  - <example>/testbench_initial.v         (step 1 - stimulus only)
  - <example>/golden_model.py             (step 2 - Python oracle)
  - <example>/test_patterns_with_golden.json (step 3 - oracle run on each pattern)
  - <example>/testbench_final.v           (step 4 - self-checking TB)
  - <example>/<module>.v                  (DUT being verified)


In [47]:
# Package everything into one zip for easy download into HW_TestbenchGen/
import zipfile, os
folders = ["mux", "adder", "sat_adder"]
with zipfile.ZipFile("tbgen_artifacts.zip", "w") as zf:
    for folder in folders:
        if not os.path.exists(folder):
            continue
        for root, dirs, files in os.walk(folder):
            for f in files:
                zf.write(os.path.join(root, f))
print("Zipped! Download tbgen_artifacts.zip from the file browser on the left.")

Zipped! Download tbgen_artifacts.zip from the file browser on the left.


---
## Prompt refinement for mux and adder final testbenches

The initial `generate_final_tb()` call for the mux and adder examples produced
Verilog that iverilog rejected (the LLM emitted SystemVerilog-only syntax:
`errors++` and chained statements via a comma operator). The simulation caught
this during verification, so we regenerate those two final testbenches with a
stricter prompt that pins the output to plain Verilog-2001 (no `++`, no comma
chaining, `begin ... end` around multi-statement if-bodies).

This is prompt refinement — the LLM is still the one producing the testbench,
and the pipeline's four-step structure is unchanged. The `sat_adder` final
testbench passed on the first attempt with the original prompt, so no retry
is needed for it.

In [44]:
import json, os, textwrap

def regen_final_tb(name, module_name):
    pwg = json.load(open(f"{name}/test_patterns_with_golden.json"))["patterns"]
    initial_tb = open(f"{name}/testbench_initial.v").read()

    prompt = textwrap.dedent(f"""\
    Here is an initial stimulus-only Verilog testbench:
```verilog
    {initial_tb}
```

    Here are the same patterns with the expected output values (JSON):
```json
    {json.dumps(pwg, indent=2)}
```

    Produce a self-checking testbench named `tb_{module_name}` with these STRICT rules so iverilog accepts it:
    - Use ONLY plain Verilog-2001 syntax. NO SystemVerilog.
    - NO `++`, `--`, or `+=` operators. Use `errors = errors + 1;` for increments.
    - EVERY `if (...)` body with more than one statement MUST be wrapped in `begin ... end`.
    - NO comma operators to chain statements.
    - `integer errors; initial errors = 0;` or `integer errors = 0;` at module scope.

    Each test should look like:
      a = ...; b = ...; cin = ...; #1;
      if (sum !== 4'bXXXX || cout !== 1'bX) begin
          $display("TEST N FAIL: ... expected sum=... cout=... got sum=%b cout=%b", sum, cout);
          errors = errors + 1;
      end else begin
          $display("TEST N PASS: ... sum=%b cout=%b", sum, cout);
      end

    At the end:
      if (errors == 0) $display("ALL TESTS PASSED");
      else             $display("%0d TEST(S) FAILED", errors);
      $finish;

    Return ONLY the Verilog testbench in one ```verilog block. No prose.
    """).strip()

    reply = ask_llm([{"role": "user", "content": prompt}])
    tb_v = extract(VBLOCK, reply, fallback_token="module ")
    open(f"{name}/testbench_final.v", "w").write(tb_v)

    rc, out = run(f"iverilog -g2012 -o tb.vvp {module_name}.v testbench_final.v && vvp tb.vvp",
                  cwd=name, timeout=60)
    passed = ("ALL TESTS PASSED" in out) and ("FAIL" not in out)
    print(f"[{name}] {'>>> PASS' if passed else '>>> FAIL'}")
    print(out.strip()[-800:])
    return passed, out

for name, module in [("mux", "mux21"), ("adder", "adder4")]:
    print(f"\n{'='*60}\nRegenerating {name}\n{'='*60}")
    ok, _ = regen_final_tb(name, module)
    if name == "mux":
        result_mux["passed"] = ok; result_mux["compiled"] = True
        result_mux["num_patterns"] = 20; result_mux["example_dir"] = "./mux"
    else:
        result_adder["passed"] = ok; result_adder["compiled"] = True
        result_adder["num_patterns"] = 20; result_adder["example_dir"] = "./adder"


Regenerating mux
[mux] >>> PASS
TEST 1 PASS: a=0 b=0 sel=0 y=0
TEST 2 PASS: a=0 b=0 sel=1 y=0
TEST 3 PASS: a=0 b=1 sel=0 y=0
TEST 4 PASS: a=0 b=1 sel=1 y=1
TEST 5 PASS: a=1 b=0 sel=0 y=1
TEST 6 PASS: a=1 b=0 sel=1 y=0
TEST 7 PASS: a=1 b=1 sel=0 y=1
TEST 8 PASS: a=1 b=1 sel=1 y=1
TEST 9 PASS: a=0 b=1 sel=0 y=0
TEST 10 PASS: a=1 b=0 sel=1 y=0
TEST 11 PASS: a=1 b=1 sel=0 y=1
TEST 12 PASS: a=0 b=0 sel=1 y=0
TEST 13 PASS: a=0 b=1 sel=1 y=1
TEST 14 PASS: a=1 b=0 sel=0 y=1
TEST 15 PASS: a=0 b=0 sel=0 y=0
TEST 16 PASS: a=1 b=1 sel=1 y=1
TEST 17 PASS: a=0 b=1 sel=0 y=0
TEST 18 PASS: a=1 b=0 sel=1 y=0
TEST 19 PASS: a=1 b=1 sel=0 y=1
TEST 20 PASS: a=0 b=1 sel=1 y=1
ALL TESTS PASSED
testbench_final.v:187: $finish called at 20 (1s)

Regenerating adder
[adder] >>> FAIL
testbench_final.v:156: syntax error
I give up.


### Second retry — adder final testbench

The first prompt refinement fixed the mux but the LLM's adder final testbench
still had a syntax error (line 155). This second attempt uses a more explicit
prompt with a full worked example in the template, and encodes the 20 patterns
as a compact `exp_sum` / `exp_cout` JSON list that the LLM fills into a fixed
TEST-block structure one-to-one.

In [45]:
# One more try for adder - with a FULL worked example in the prompt
import json, textwrap

pwg = json.load(open("adder/test_patterns_with_golden.json"))["patterns"]
# Build a concise JSON of just the patterns (not the giant dump)
pat_summary = [{"a": p["inputs"]["a"], "b": p["inputs"]["b"], "cin": p["inputs"]["cin"],
                "exp_sum": p["expected"]["sum"], "exp_cout": p["expected"]["cout"]}
               for p in pwg]

prompt = textwrap.dedent(f"""\
Generate a plain Verilog-2001 self-checking testbench for a 4-bit adder with carry-in.
Module is instantiated as:  adder4 dut(.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));

Use EXACTLY this template structure, filling in ALL 20 patterns from the JSON below:

```verilog
module tb_adder4;
    reg [3:0] a, b;
    reg cin;
    wire [3:0] sum;
    wire cout;
    integer errors;
    adder4 dut(.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));
    initial begin
        errors = 0;

        // TEST 1
        a = 4'd0; b = 4'd0; cin = 1'b0; #1;
        if (sum !== 4'd0 || cout !== 1'b0) begin
            $display("TEST 1 FAIL: a=%0d b=%0d cin=%0d expected sum=0 cout=0 got sum=%0d cout=%0d", a, b, cin, sum, cout);
            errors = errors + 1;
        end else begin
            $display("TEST 1 PASS: a=%0d b=%0d cin=%0d sum=%0d cout=%0d", a, b, cin, sum, cout);
        end

        // TEST 2 ... same structure ...

        if (errors == 0) $display("ALL TESTS PASSED");
        else             $display("%0d TEST(S) FAILED", errors);
        $finish;
    end
endmodule
```

RULES:
- NO SystemVerilog (no ++, no +=, no comma operators).
- EVERY if-body MUST use begin ... end.
- Separate each statement with ;
- Follow the template EXACTLY.

Patterns to encode (one TEST block each, numbered 1..{len(pat_summary)}):
```json
{json.dumps(pat_summary, indent=1)}
```

Return ONLY the complete Verilog in one ```verilog block. No prose.
""").strip()

reply = ask_llm([{"role": "user", "content": prompt}], max_tokens=4000)
tb_v = extract(VBLOCK, reply, fallback_token="module ")
open("adder/testbench_final.v", "w").write(tb_v)

rc, out = run("iverilog -g2012 -o tb.vvp adder4.v testbench_final.v && vvp tb.vvp",
              cwd="adder", timeout=60)
passed = ("ALL TESTS PASSED" in out) and ("FAIL" not in out)
print(f"[adder] {'>>> PASS' if passed else '>>> FAIL'}")
print(out.strip()[-1200:])

result_adder["passed"] = passed; result_adder["compiled"] = True
result_adder["num_patterns"] = 20; result_adder["example_dir"] = "./adder"

[adder] >>> PASS
TEST 1 PASS: a=0 b=0 cin=0 sum=0 cout=0
TEST 2 PASS: a=1 b=1 cin=0 sum=2 cout=0
TEST 3 PASS: a=2 b=3 cin=0 sum=5 cout=0
TEST 4 PASS: a=15 b=1 cin=0 sum=0 cout=1
TEST 5 PASS: a=15 b=15 cin=0 sum=14 cout=1
TEST 6 PASS: a=7 b=8 cin=1 sum=0 cout=1
TEST 7 PASS: a=10 b=5 cin=1 sum=0 cout=1
TEST 8 PASS: a=0 b=15 cin=1 sum=0 cout=1
TEST 9 PASS: a=15 b=0 cin=1 sum=0 cout=1
TEST 10 PASS: a=12 b=12 cin=0 sum=8 cout=1
TEST 11 PASS: a=3 b=3 cin=1 sum=7 cout=0
TEST 12 PASS: a=5 b=5 cin=0 sum=10 cout=0
TEST 13 PASS: a=9 b=9 cin=1 sum=3 cout=1
TEST 14 PASS: a=14 b=1 cin=0 sum=15 cout=0
TEST 15 PASS: a=6 b=6 cin=1 sum=13 cout=0
TEST 16 PASS: a=0 b=0 cin=1 sum=1 cout=0
TEST 17 PASS: a=15 b=15 cin=1 sum=15 cout=1
TEST 18 PASS: a=4 b=3 cin=0 sum=7 cout=0
TEST 19 PASS: a=8 b=8 cin=0 sum=0 cout=1
TEST 20 PASS: a=7 b=7 cin=1 sum=15 cout=0
ALL TESTS PASSED
testbench_final.v:193: $finish called at 20 (1s)
